# 🏦 QUANTUM TRADING SYSTEM — v2.0 Professional Edition
### Multi-Timeframe (1D→4H→1H→15m) | Technical + News Sentiment Analysis
**Inspired by:** George Soros (Macro Risk) | Stanley Druckenmiller (Asymmetric Bets) | Michael Marcus (Technical Precision)

---
**Exchange:** Bitget | **Pair:** BTC/USDT | **Mode:** Spot

> *"Risk comes from not knowing what you're doing."* — George Soros
>
> *"The markets are always changing, and the successful trader needs to adapt to those changes."* — Michael Marcus

In [ ]:
# ─── INSTALL DEPENDENCIES ───────────────────────────────────────────────────
!pip install ccxt ta mplfinance pandas numpy matplotlib pytz requests beautifulsoup4 --quiet

In [ ]:
# ─── IMPORTS & GLOBAL CONFIG ────────────────────────────────────────────────
import ccxt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from mplfinance.original_flavor import candlestick_ohlc
import ta
import time
import json
import os
import re
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from IPython.display import clear_output, display
import pytz
import warnings
warnings.filterwarnings('ignore')

# ─── TIMEZONE ───────────────────────────────────────────────────────────────
WIB = pytz.timezone('Asia/Jakarta')

# ─── STRATEGY CONFIG ────────────────────────────────────────────────────────
CONFIG = {
    'symbol':        'BTC/USDT',

    # Multi-Timeframe (HTF → LTF)
    'tf_daily':      '1d',   # Trend utama
    'tf_4h':         '4h',   # Konfirmasi menengah
    'tf_1h':         '1h',   # Konfirmasi entry
    'tf_main':       '15m',  # Entry presisi
    'limit_daily':   100,
    'limit_4h':      200,
    'limit_1h':      200,
    'limit_main':    300,

    # Indikator
    'bb_window':     20,
    'bb_std':        2.0,
    'rsi_window':    14,
    'ema_fast':      9,
    'ema_slow':      21,
    'ema_trend':     55,
    'macd_fast':     12,
    'macd_slow':     26,
    'macd_signal':   9,
    'atr_window':    14,
    'atr_tp_mult':   2.5,
    'atr_sl_mult':   1.2,
    'vol_ma':        20,
    'fib_lookback':  50,

    # Scoring
    'rsi_oversold':  35,
    'rsi_overbought':65,
    'min_score':     4,   # Dari maks 10 (6 teknikal + 2 MTF + 2 berita)

    # Loop
    'update_sec':    300,
    'signal_history_file': 'signal_history_v2.json',

    # News
    'news_max_age_hours': 6,   # Berita yang dianggap relevan (maks 6 jam lalu)
    'news_max_items':     30,  # Maksimum berita yang diambil
}

# ─── EXCHANGE SETUP ─────────────────────────────────────────────────────────
exchange = ccxt.bitget({
    'enableRateLimit': True,
    'options': {'defaultType': 'spot'}
})

print('✅ Configuration loaded.')
print(f"📌 Symbol : {CONFIG['symbol']}")
print(f"📊 TF     : 1D → 4H → 1H → 15m")
print(f"🎯 Min Score: {CONFIG['min_score']}/10 (6 Teknikal + 2 MTF + 2 News)")

In [ ]:
# ─── SIGNAL HISTORY MANAGER ─────────────────────────────────────────────────

class SignalHistory:
    """Persistent signal log — survives kernel restarts via JSON."""

    def __init__(self, filepath):
        self.filepath = filepath
        self.records  = self._load()

    def _load(self):
        if os.path.exists(self.filepath):
            with open(self.filepath, 'r') as f:
                return json.load(f)
        return []

    def _save(self):
        with open(self.filepath, 'w') as f:
            json.dump(self.records, f, indent=2, default=str)

    def add(self, signal_dict):
        key = f"{signal_dict['timestamp']}_{signal_dict['type']}"
        for r in self.records:
            if f"{r['timestamp']}_{r['type']}" == key:
                return False
        signal_dict['logged_at'] = datetime.now(WIB).strftime('%Y-%m-%d %H:%M:%S')
        self.records.append(signal_dict)
        self.records = self.records[-200:]
        self._save()
        return True

    def get_recent(self, n=20):
        return self.records[-n:]

    def to_dataframe(self):
        if not self.records:
            return pd.DataFrame()
        df = pd.DataFrame(self.records)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        return df.sort_values('timestamp', ascending=False)

HISTORY = SignalHistory(CONFIG['signal_history_file'])
print(f"📂 Signal history loaded: {len(HISTORY.records)} records found.")

In [ ]:
# ─── NEWS SENTIMENT ENGINE ───────────────────────────────────────────────────
#
# Mengambil berita dari CryptoPanic RSS feed (gratis, tanpa API key)
# dan CoinTelegraph RSS feed sebagai sumber kedua.
# Setiap judul berita di-score berdasarkan kata kunci bullish / bearish.

BULLISH_KEYWORDS = [
    'surge', 'rally', 'bull', 'bullish', 'pump', 'moon', 'breakout', 'all-time high',
    'ath', 'gain', 'rise', 'adoption', 'institutional', 'etf approved', 'buy',
    'upgrade', 'record', 'green', 'recovery', 'rebound', 'support', 'halving',
    'accumulate', 'inflow', 'positive', 'invest', 'listing', 'partnership',
    'naik', 'bullish', 'beli', 'pembelian', 'investasi', 'rekor'
]

BEARISH_KEYWORDS = [
    'crash', 'dump', 'bear', 'bearish', 'plunge', 'drop', 'fall', 'decline',
    'ban', 'hack', 'exploit', 'lawsuit', 'sec', 'regulation', 'restrict',
    'warning', 'fraud', 'scam', 'sell', 'short', 'fear', 'loss', 'red',
    'outflow', 'resistance', 'correction', 'liquidation', 'bankruptcy',
    'turun', 'jual', 'larangan', 'hack', 'gagal', 'runtuh'
]

HIGH_IMPACT_KEYWORDS = [
    'fed', 'federal reserve', 'interest rate', 'inflation', 'cpi', 'fomc',
    'sec', 'etf', 'blackrock', 'fidelity', 'spot btc', 'bitcoin etf',
    'all-time high', 'ath', 'halving', 'ban', 'hack', 'exploit', 'bankruptcy'
]

NEWS_SOURCES = [
    {
        'name': 'CryptoPanic',
        'url': 'https://cryptopanic.com/news/bitcoin/rss/',
        'type': 'rss'
    },
    {
        'name': 'CoinTelegraph',
        'url': 'https://cointelegraph.com/rss/tag/bitcoin',
        'type': 'rss'
    },
    {
        'name': 'Bitcoin Magazine',
        'url': 'https://bitcoinmagazine.com/.rss/full/',
        'type': 'rss'
    },
]


def fetch_news():
    """Ambil berita dari semua sumber RSS, parsing, dan score sentiment."""
    all_news = []
    now_utc  = datetime.utcnow().replace(tzinfo=pytz.utc)
    cutoff   = now_utc - timedelta(hours=CONFIG['news_max_age_hours'])

    headers = {
        'User-Agent': 'Mozilla/5.0 (compatible; TradingBot/2.0)'
    }

    for src in NEWS_SOURCES:
        try:
            resp = requests.get(src['url'], headers=headers, timeout=10)
            if resp.status_code != 200:
                continue
            soup = BeautifulSoup(resp.content, 'xml')
            items = soup.find_all('item')

            for item in items[:CONFIG['news_max_items']]:
                title = item.find('title')
                pubdate = item.find('pubDate')
                link = item.find('link')
                desc = item.find('description')

                if not title:
                    continue

                title_text = title.get_text(strip=True)
                desc_text  = desc.get_text(strip=True)[:300] if desc else ''
                full_text  = (title_text + ' ' + desc_text).lower()

                # Parse published date
                pub_dt = None
                if pubdate:
                    try:
                        from email.utils import parsedate_to_datetime
                        pub_dt = parsedate_to_datetime(pubdate.get_text(strip=True))
                        if pub_dt.tzinfo is None:
                            pub_dt = pub_dt.replace(tzinfo=pytz.utc)
                    except:
                        pub_dt = now_utc
                else:
                    pub_dt = now_utc

                # Skip berita lama
                if pub_dt < cutoff:
                    continue

                # Hitung sentiment score
                bull_count = sum(1 for kw in BULLISH_KEYWORDS if kw in full_text)
                bear_count = sum(1 for kw in BEARISH_KEYWORDS if kw in full_text)
                high_impact = any(kw in full_text for kw in HIGH_IMPACT_KEYWORDS)

                net_score = bull_count - bear_count
                if high_impact:
                    net_score = net_score * 2  # Berita high-impact bobotnya 2x

                sentiment = 'BULLISH' if net_score > 0 else 'BEARISH' if net_score < 0 else 'NEUTRAL'
                age_hours = (now_utc - pub_dt).total_seconds() / 3600

                all_news.append({
                    'source':      src['name'],
                    'title':       title_text,
                    'sentiment':   sentiment,
                    'net_score':   net_score,
                    'high_impact': high_impact,
                    'age_hours':   round(age_hours, 1),
                    'pub_dt':      pub_dt,
                    'url':         link.get_text(strip=True) if link else '',
                })

        except Exception as e:
            print(f"  ⚠️  Gagal ambil berita dari {src['name']}: {e}")
            continue

    # Urutkan berita terbaru dulu
    all_news.sort(key=lambda x: x['pub_dt'], reverse=True)
    return all_news


def aggregate_news_sentiment(news_list):
    """
    Agregat semua berita menjadi satu sentiment score:
    - Kembalikan: (sentiment_label, news_score 0-2, summary_str)
    """
    if not news_list:
        return 'NEUTRAL', 0, 'Tidak ada berita relevan ditemukan.'

    total     = sum(n['net_score'] for n in news_list)
    bull_count = sum(1 for n in news_list if n['sentiment'] == 'BULLISH')
    bear_count = sum(1 for n in news_list if n['sentiment'] == 'BEARISH')
    high_impact_bull = sum(1 for n in news_list if n['high_impact'] and n['net_score'] > 0)
    high_impact_bear = sum(1 for n in news_list if n['high_impact'] and n['net_score'] < 0)

    # News score untuk sinyal: 0, 1, atau 2
    if total >= 4 or high_impact_bull >= 1:
        sentiment_label = 'VERY BULLISH 🚀'
        news_score_bull = 2
        news_score_bear = 0
    elif total > 0:
        sentiment_label = 'BULLISH 🟢'
        news_score_bull = 1
        news_score_bear = 0
    elif total <= -4 or high_impact_bear >= 1:
        sentiment_label = 'VERY BEARISH 💀'
        news_score_bull = 0
        news_score_bear = 2
    elif total < 0:
        sentiment_label = 'BEARISH 🔴'
        news_score_bull = 0
        news_score_bear = 1
    else:
        sentiment_label = 'NEUTRAL ⚪'
        news_score_bull = 0
        news_score_bear = 0

    summary = (
        f"{len(news_list)} berita | 🟢 {bull_count} Bullish | 🔴 {bear_count} Bearish | "
        f"💥 {high_impact_bull+high_impact_bear} High-Impact | Net Score: {total:+d}"
    )

    return sentiment_label, news_score_bull, news_score_bear, summary


print('✅ News Sentiment Engine ready.')

In [ ]:
# ─── INDICATOR ENGINE ───────────────────────────────────────────────────────

def compute_indicators(df):
    """Hitung full indikator suite pada dataframe OHLCV."""

    # Bollinger Bands
    bb = ta.volatility.BollingerBands(
        df['close'], window=CONFIG['bb_window'], window_dev=CONFIG['bb_std']
    )
    df['bb_upper'] = bb.bollinger_hband()
    df['bb_mid']   = bb.bollinger_mavg()
    df['bb_lower'] = bb.bollinger_lband()
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_mid']

    # RSI
    df['rsi'] = ta.momentum.RSIIndicator(
        df['close'], window=CONFIG['rsi_window']
    ).rsi()

    # EMAs
    df['ema_fast']  = ta.trend.EMAIndicator(df['close'], window=CONFIG['ema_fast']).ema_indicator()
    df['ema_slow']  = ta.trend.EMAIndicator(df['close'], window=CONFIG['ema_slow']).ema_indicator()
    df['ema_trend'] = ta.trend.EMAIndicator(df['close'], window=CONFIG['ema_trend']).ema_indicator()

    # MACD
    macd = ta.trend.MACD(
        df['close'],
        window_fast=CONFIG['macd_fast'],
        window_slow=CONFIG['macd_slow'],
        window_sign=CONFIG['macd_signal']
    )
    df['macd']        = macd.macd()
    df['macd_signal'] = macd.macd_signal()
    df['macd_hist']   = macd.macd_diff()

    # ATR
    df['atr'] = ta.volatility.AverageTrueRange(
        df['high'], df['low'], df['close'], window=CONFIG['atr_window']
    ).average_true_range()

    # Volume MA
    df['vol_ma']    = df['volume'].rolling(CONFIG['vol_ma']).mean()
    df['vol_ratio'] = df['volume'] / df['vol_ma']

    return df


def compute_fibonacci(df):
    """Hitung level Fibonacci retracement dari swing terakhir."""
    lookback   = CONFIG['fib_lookback']
    swing_high = df['high'].rolling(lookback).max().iloc[-1]
    swing_low  = df['low'].rolling(lookback).min().iloc[-1]
    diff = swing_high - swing_low
    levels = {
        '0.0':   swing_high,
        '0.236': swing_high - 0.236 * diff,
        '0.382': swing_high - 0.382 * diff,
        '0.500': swing_high - 0.500 * diff,
        '0.618': swing_high - 0.618 * diff,
        '0.786': swing_high - 0.786 * diff,
        '1.0':   swing_low,
    }
    return levels, swing_high, swing_low


def get_htf_trend(df_daily, df_4h, df_1h):
    """
    Analisis tren dari Higher Timeframe (HTF).
    Return: (htf_bias, htf_reasons)
      htf_bias = 'BULLISH' | 'BEARISH' | 'NEUTRAL'
    """
    reasons = []
    bull_count = 0
    bear_count = 0

    # ── DAILY ────────────────────────────────────────────────────────────────
    if len(df_daily) >= 55:
        d = df_daily.iloc[-1]
        d_ema55 = ta.trend.EMAIndicator(df_daily['close'], window=55).ema_indicator().iloc[-1]
        d_rsi   = ta.momentum.RSIIndicator(df_daily['close'], window=14).rsi().iloc[-1]
        if d['close'] > d_ema55:
            bull_count += 1; reasons.append('1D: Harga > EMA55 ↑')
        else:
            bear_count += 1; reasons.append('1D: Harga < EMA55 ↓')
        if d_rsi > 50:
            bull_count += 1; reasons.append(f'1D: RSI {d_rsi:.0f} > 50 ↑')
        else:
            bear_count += 1; reasons.append(f'1D: RSI {d_rsi:.0f} < 50 ↓')

    # ── 4H ───────────────────────────────────────────────────────────────────
    if len(df_4h) >= 55:
        h4 = df_4h.iloc[-1]
        h4_ema21  = ta.trend.EMAIndicator(df_4h['close'], window=21).ema_indicator().iloc[-1]
        h4_ema55  = ta.trend.EMAIndicator(df_4h['close'], window=55).ema_indicator().iloc[-1]
        h4_macd   = ta.trend.MACD(df_4h['close']).macd_diff().iloc[-1]
        if h4_ema21 > h4_ema55:
            bull_count += 1; reasons.append('4H: EMA21 > EMA55 ↑')
        else:
            bear_count += 1; reasons.append('4H: EMA21 < EMA55 ↓')
        if h4_macd > 0:
            bull_count += 1; reasons.append('4H: MACD bullish ↑')
        else:
            bear_count += 1; reasons.append('4H: MACD bearish ↓')

    # ── 1H ───────────────────────────────────────────────────────────────────
    if len(df_1h) >= 21:
        h1 = df_1h.iloc[-1]
        h1_ema21 = ta.trend.EMAIndicator(df_1h['close'], window=21).ema_indicator().iloc[-1]
        if h1['close'] > h1_ema21:
            bull_count += 1; reasons.append('1H: Harga > EMA21 ↑')
        else:
            bear_count += 1; reasons.append('1H: Harga < EMA21 ↓')

    if bull_count > bear_count:
        htf_bias = 'BULLISH'
    elif bear_count > bull_count:
        htf_bias = 'BEARISH'
    else:
        htf_bias = 'NEUTRAL'

    return htf_bias, bull_count, bear_count, reasons


print('✅ Indicator engine ready.')

In [ ]:
# ─── SIGNAL SCORING ENGINE ───────────────────────────────────────────────────
#
# Total skor maks = 10:
#   6 poin → Teknikal (BB, RSI, Fibonacci, Volume, MACD, EMA)
#   2 poin → Multi-Timeframe (konfirmasi HTF)
#   2 poin → News Sentiment

def score_signal(row, df, idx_pos, signal_type, fib_levels,
                 htf_bias, htf_bull, htf_bear,
                 news_score_bull, news_score_bear):
    score   = 0
    reasons = []

    if signal_type == 'BUY':
        # ── TEKNIKAL (0–6) ───────────────────────────────────────────────────
        if row['close'] < row['bb_lower']:
            score += 1; reasons.append('Harga < Lower BB')
        if row['rsi'] < CONFIG['rsi_oversold']:
            score += 1; reasons.append(f"RSI({row['rsi']:.0f}) oversold")
        fib_support = min(fib_levels['0.618'], fib_levels['0.786'])
        fib_zone    = max(fib_levels['0.618'], fib_levels['0.786'])
        if fib_support * 0.998 <= row['close'] <= fib_zone * 1.002:
            score += 1; reasons.append('Dekat support Fib 0.618/0.786')
        if row['vol_ratio'] > 1.2:
            score += 1; reasons.append(f"Volume spike {row['vol_ratio']:.1f}x")
        if idx_pos > 0 and row['macd_hist'] > df['macd_hist'].iloc[idx_pos - 1]:
            score += 1; reasons.append('MACD hist naik')
        if row['close'] > row['ema_trend']:
            score += 1; reasons.append('Harga > EMA55 (uptrend)')

        # ── MULTI-TIMEFRAME (0–2) ─────────────────────────────────────────────
        if htf_bias == 'BULLISH':
            mtf_pts = min(htf_bull, 2)
            score  += mtf_pts
            reasons.append(f'HTF BULLISH [{htf_bull}vs{htf_bear}] (+{mtf_pts})')
        elif htf_bias == 'BEARISH':
            reasons.append(f'HTF BEARISH [{htf_bull}vs{htf_bear}] (peringatan!)')

        # ── NEWS (0–2) ────────────────────────────────────────────────────────
        if news_score_bull > 0:
            score += news_score_bull
            reasons.append(f'News sentiment BULLISH (+{news_score_bull})')
        elif news_score_bear > 0:
            reasons.append(f'News sentiment BEARISH (−{news_score_bear}, peringatan!)')

    elif signal_type == 'SELL':
        # ── TEKNIKAL (0–6) ───────────────────────────────────────────────────
        if row['close'] > row['bb_upper']:
            score += 1; reasons.append('Harga > Upper BB')
        if row['rsi'] > CONFIG['rsi_overbought']:
            score += 1; reasons.append(f"RSI({row['rsi']:.0f}) overbought")
        fib_res_low  = min(fib_levels['0.0'], fib_levels['0.236'])
        fib_res_high = max(fib_levels['0.0'], fib_levels['0.236'])
        if fib_res_low * 0.998 <= row['close'] <= fib_res_high * 1.002:
            score += 1; reasons.append('Dekat resistance Fib 0.0/0.236')
        if row['vol_ratio'] > 1.2:
            score += 1; reasons.append(f"Volume spike {row['vol_ratio']:.1f}x")
        if idx_pos > 0 and row['macd_hist'] < df['macd_hist'].iloc[idx_pos - 1]:
            score += 1; reasons.append('MACD hist turun')
        if row['close'] < row['ema_trend']:
            score += 1; reasons.append('Harga < EMA55 (downtrend)')

        # ── MULTI-TIMEFRAME (0–2) ─────────────────────────────────────────────
        if htf_bias == 'BEARISH':
            mtf_pts = min(htf_bear, 2)
            score  += mtf_pts
            reasons.append(f'HTF BEARISH [{htf_bull}vs{htf_bear}] (+{mtf_pts})')
        elif htf_bias == 'BULLISH':
            reasons.append(f'HTF BULLISH [{htf_bull}vs{htf_bear}] (peringatan!)')

        # ── NEWS (0–2) ────────────────────────────────────────────────────────
        if news_score_bear > 0:
            score += news_score_bear
            reasons.append(f'News sentiment BEARISH (+{news_score_bear})')
        elif news_score_bull > 0:
            reasons.append(f'News sentiment BULLISH (−{news_score_bull}, peringatan!)')

    return score, reasons


def detect_signals(df, fib_levels, htf_bias, htf_bull, htf_bear,
                   news_score_bull, news_score_bear):
    """Scan semua candle, score sinyal, kembalikan sinyal terkonfirmasi."""
    confirmed = []

    for i in range(CONFIG['bb_window'], len(df)):
        row = df.iloc[i]

        for sig_type in ['BUY', 'SELL']:
            if sig_type == 'BUY'  and not (row['close'] < row['bb_lower']):
                continue
            if sig_type == 'SELL' and not (row['close'] > row['bb_upper']):
                continue

            score, reasons = score_signal(
                row, df, i, sig_type, fib_levels,
                htf_bias, htf_bull, htf_bear,
                news_score_bull, news_score_bear
            )

            if score >= CONFIG['min_score']:
                atr = row['atr'] if not np.isnan(row['atr']) else row['close'] * 0.01
                if sig_type == 'BUY':
                    tp = row['close'] + atr * CONFIG['atr_tp_mult']
                    sl = row['close'] - atr * CONFIG['atr_sl_mult']
                    rr = (tp - row['close']) / (row['close'] - sl)
                else:
                    tp = row['close'] - atr * CONFIG['atr_tp_mult']
                    sl = row['close'] + atr * CONFIG['atr_sl_mult']
                    rr = (row['close'] - tp) / (sl - row['close'])

                confirmed.append((df.index[i], {
                    'timestamp': str(df.index[i]),
                    'type':      sig_type,
                    'close':     round(float(row['close']), 4),
                    'tp':        round(float(tp), 4),
                    'sl':        round(float(sl), 4),
                    'rr':        round(float(rr), 2),
                    'rsi':       round(float(row['rsi']), 2),
                    'atr':       round(float(atr), 4),
                    'vol_ratio': round(float(row['vol_ratio']), 2),
                    'score':     score,
                    'htf_bias':  htf_bias,
                    'reasons':   reasons,
                }))

    return confirmed


print('✅ Signal scoring engine ready. (Max score: 10)')

In [ ]:
# ─── CHART ENGINE ───────────────────────────────────────────────────────────

COLORS = {
    'bg':     '#0d1117',
    'panel':  '#161b22',
    'grid':   '#21262d',
    'text':   '#c9d1d9',
    'green':  '#3fb950',
    'red':    '#f85149',
    'yellow': '#d29922',
    'blue':   '#58a6ff',
    'purple': '#bc8cff',
    'orange': '#ffa657',
    'teal':   '#39d353',
    'dim':    '#6e7681',
}

def setup_ax(ax):
    ax.set_facecolor(COLORS['panel'])
    ax.tick_params(colors=COLORS['text'], labelsize=7)
    for sp in ax.spines.values():
        sp.set_edgecolor(COLORS['grid'])
    ax.grid(True, color=COLORS['grid'], linestyle='--', linewidth=0.4, alpha=0.6)


def plot_chart(df, confirmed_signals, fib_levels, history_df, news_list,
               sentiment_label, htf_bias, htf_reasons, now_str):
    """5-panel chart: Price | RSI | MACD | Volume | News Feed + History Table"""

    N = 120
    df_plot = df.iloc[-N:].copy()
    df_plot['date_num'] = mdates.date2num(df_plot.index.to_pydatetime())

    fig = plt.figure(figsize=(22, 16), facecolor=COLORS['bg'])
    gs  = gridspec.GridSpec(
        5, 2,
        height_ratios=[4, 1.2, 1.2, 1.0, 1.4],
        width_ratios=[2.6, 1],
        hspace=0.06, wspace=0.22,
        left=0.05, right=0.97, top=0.93, bottom=0.05
    )

    ax_price = fig.add_subplot(gs[0, 0])
    ax_rsi   = fig.add_subplot(gs[1, 0], sharex=ax_price)
    ax_macd  = fig.add_subplot(gs[2, 0], sharex=ax_price)
    ax_vol   = fig.add_subplot(gs[3, 0], sharex=ax_price)
    ax_news  = fig.add_subplot(gs[4, 0])
    ax_table = fig.add_subplot(gs[:, 1])

    for ax in [ax_price, ax_rsi, ax_macd, ax_vol]:
        setup_ax(ax)
    ax_news.set_facecolor(COLORS['panel'])
    for sp in ax_news.spines.values():
        sp.set_edgecolor(COLORS['grid'])
    ax_table.set_facecolor(COLORS['panel'])
    for sp in ax_table.spines.values():
        sp.set_edgecolor(COLORS['grid'])

    # ── PANEL 1: CANDLESTICK ──────────────────────────────────────────────────
    ohlc = df_plot[['date_num','open','high','low','close']].values
    candlestick_ohlc(ax_price, ohlc, width=0.003,
                     colorup=COLORS['green'], colordown=COLORS['red'], alpha=0.85)

    ax_price.plot(df_plot.index, df_plot['bb_upper'], color=COLORS['purple'],  lw=0.9, ls='--', alpha=0.8, label='Upper BB')
    ax_price.plot(df_plot.index, df_plot['bb_mid'],   color=COLORS['blue'],    lw=0.9, ls='--', alpha=0.8, label='Mid BB')
    ax_price.plot(df_plot.index, df_plot['bb_lower'], color=COLORS['orange'],  lw=0.9, ls='--', alpha=0.8, label='Lower BB')
    ax_price.fill_between(df_plot.index, df_plot['bb_lower'], df_plot['bb_upper'],
                          alpha=0.04, color=COLORS['purple'])
    ax_price.plot(df_plot.index, df_plot['ema_fast'],  color=COLORS['teal'],   lw=0.8, label=f"EMA{CONFIG['ema_fast']}")
    ax_price.plot(df_plot.index, df_plot['ema_slow'],  color=COLORS['yellow'], lw=0.8, label=f"EMA{CONFIG['ema_slow']}")
    ax_price.plot(df_plot.index, df_plot['ema_trend'], color=COLORS['red'],    lw=1.0, alpha=0.7, label=f"EMA{CONFIG['ema_trend']}")

    fib_colors = ['#58a6ff','#3fb950','#ffa657','#d29922','#bc8cff','#f85149','#39d353']
    for (lvl, price), fc in zip(fib_levels.items(), fib_colors):
        ax_price.axhline(price, color=fc, lw=0.6, ls=':', alpha=0.5)
        ax_price.text(df_plot.index[-1], price, f' Fib {lvl} ({price:,.0f})',
                      color=fc, fontsize=6.5, va='center',
                      bbox=dict(boxstyle='round,pad=0.1', facecolor=COLORS['bg'], alpha=0.6))

    plot_range = set(df_plot.index)
    for ts, sig in confirmed_signals:
        if ts not in plot_range:
            continue
        price = sig['close']
        stars = '★' * min(sig['score'], 10)
        if sig['type'] == 'BUY':
            ax_price.scatter(ts, price * 0.9985, marker='^', s=140, color=COLORS['green'], zorder=10)
            ax_price.annotate(f" BUY\n {price:,.0f}\n{stars}",
                              (ts, price * 0.997), fontsize=6, color=COLORS['green'],
                              ha='center', va='top',
                              bbox=dict(boxstyle='round,pad=0.2', fc=COLORS['bg'], alpha=0.7))
        else:
            ax_price.scatter(ts, price * 1.0015, marker='v', s=140, color=COLORS['red'], zorder=10)
            ax_price.annotate(f" SELL\n {price:,.0f}\n{stars}",
                              (ts, price * 1.003), fontsize=6, color=COLORS['red'],
                              ha='center', va='bottom',
                              bbox=dict(boxstyle='round,pad=0.2', fc=COLORS['bg'], alpha=0.7))

    # HTF bias label
    htf_color = COLORS['green'] if htf_bias == 'BULLISH' else COLORS['red'] if htf_bias == 'BEARISH' else COLORS['yellow']
    ax_price.text(0.01, 0.97, f"HTF Bias: {htf_bias}",
                  transform=ax_price.transAxes, fontsize=8, color=htf_color,
                  fontweight='bold', va='top',
                  bbox=dict(boxstyle='round,pad=0.3', facecolor=COLORS['bg'], alpha=0.8))

    ax_price.set_ylabel('Price (USDT)', color=COLORS['text'], fontsize=8)
    ax_price.legend(loc='upper right', fontsize=6.5, framealpha=0.3,
                    facecolor=COLORS['bg'], labelcolor=COLORS['text'])
    plt.setp(ax_price.get_xticklabels(), visible=False)

    # ── PANEL 2: RSI ──────────────────────────────────────────────────────────
    ax_rsi.plot(df_plot.index, df_plot['rsi'], color=COLORS['blue'], lw=0.9)
    ax_rsi.axhline(70, color=COLORS['red'],   lw=0.7, ls='--', alpha=0.6)
    ax_rsi.axhline(30, color=COLORS['green'], lw=0.7, ls='--', alpha=0.6)
    ax_rsi.axhline(50, color=COLORS['dim'],   lw=0.5, ls=':')
    ax_rsi.fill_between(df_plot.index, df_plot['rsi'], 30, where=(df_plot['rsi'] < 30),
                        color=COLORS['green'], alpha=0.15)
    ax_rsi.fill_between(df_plot.index, df_plot['rsi'], 70, where=(df_plot['rsi'] > 70),
                        color=COLORS['red'], alpha=0.15)
    ax_rsi.set_ylim(0, 100)
    ax_rsi.set_ylabel('RSI', color=COLORS['text'], fontsize=8)
    ax_rsi.text(df_plot.index[-1], df_plot['rsi'].iloc[-1],
                f" {df_plot['rsi'].iloc[-1]:.1f}",
                color=COLORS['blue'], fontsize=7, va='center')
    plt.setp(ax_rsi.get_xticklabels(), visible=False)

    # ── PANEL 3: MACD ─────────────────────────────────────────────────────────
    ax_macd.plot(df_plot.index, df_plot['macd'],        color=COLORS['blue'],   lw=0.9, label='MACD')
    ax_macd.plot(df_plot.index, df_plot['macd_signal'], color=COLORS['orange'], lw=0.9, label='Signal')
    hist_colors = [COLORS['green'] if v >= 0 else COLORS['red'] for v in df_plot['macd_hist']]
    ax_macd.bar(df_plot.index, df_plot['macd_hist'], color=hist_colors, width=0.003, alpha=0.7)
    ax_macd.axhline(0, color=COLORS['dim'], lw=0.5)
    ax_macd.set_ylabel('MACD', color=COLORS['text'], fontsize=8)
    ax_macd.legend(loc='upper left', fontsize=6, framealpha=0.2,
                   facecolor=COLORS['bg'], labelcolor=COLORS['text'])
    plt.setp(ax_macd.get_xticklabels(), visible=False)

    # ── PANEL 4: VOLUME ───────────────────────────────────────────────────────
    vol_colors = [COLORS['green'] if c >= o else COLORS['red']
                  for c, o in zip(df_plot['close'], df_plot['open'])]
    ax_vol.bar(df_plot.index, df_plot['volume'], color=vol_colors, width=0.003, alpha=0.7)
    ax_vol.plot(df_plot.index, df_plot['vol_ma'], color=COLORS['blue'], lw=0.9, label='Vol MA20')
    ax_vol.set_ylabel('Volume', color=COLORS['text'], fontsize=8)
    ax_vol.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %H:%M', tz=WIB))
    ax_vol.tick_params(axis='x', rotation=30, labelsize=6.5)
    ax_vol.legend(loc='upper left', fontsize=6, framealpha=0.2,
                  facecolor=COLORS['bg'], labelcolor=COLORS['text'])
    plt.setp(ax_vol.get_xticklabels(), visible=False)

    # ── PANEL 5: NEWS FEED ────────────────────────────────────────────────────
    ax_news.axis('off')
    ax_news.set_title(f'📰 Berita Terbaru  |  Sentiment: {sentiment_label}',
                      color=COLORS['text'], fontsize=8, fontweight='bold', pad=4)

    news_to_show = news_list[:8]
    y_pos = 0.95
    for n in news_to_show:
        clr = COLORS['green'] if n['sentiment'] == 'BULLISH' else \
              COLORS['red']   if n['sentiment'] == 'BEARISH' else COLORS['dim']
        icon = '🟢' if n['sentiment'] == 'BULLISH' else '🔴' if n['sentiment'] == 'BEARISH' else '⚪'
        hi   = ' 💥' if n['high_impact'] else ''
        title_short = n['title'][:80] + ('…' if len(n['title']) > 80 else '')
        line = f"{icon}{hi} [{n['age_hours']:.1f}j | {n['source']}] {title_short}"
        ax_news.text(0.01, y_pos, line,
                     transform=ax_news.transAxes,
                     color=clr, fontsize=6.5, va='top', wrap=True)
        y_pos -= 0.125
        if y_pos < 0:
            break

    if not news_to_show:
        ax_news.text(0.5, 0.5, 'Tidak ada berita terbaru.',
                     transform=ax_news.transAxes, ha='center', va='center',
                     color=COLORS['dim'], fontsize=9)

    # ── PANEL KANAN: HISTORY TABLE ────────────────────────────────────────────
    ax_table.axis('off')
    ax_table.set_title('📋 Signal History (Last 15)', color=COLORS['text'],
                        fontsize=9, fontweight='bold', pad=6)

    if not history_df.empty:
        recent = history_df.head(15)
        rows, col_colors = [], []
        for _, r in recent.iterrows():
            ts_str = pd.to_datetime(r['timestamp']).strftime('%d/%m %H:%M')
            rows.append([
                ts_str,
                r['type'],
                f"{float(r['close']):,.0f}",
                f"{float(r['tp']):,.0f}",
                f"{float(r['sl']):,.0f}",
                f"{float(r['rr']):.1f}R",
                f"{int(r['score'])}/10",
                r.get('htf_bias', '-')[:4],
            ])
            c = COLORS['green'] if r['type'] == 'BUY' else COLORS['red']
            col_colors.append(['#1c2128', c, '#1c2128', '#1c2128', '#1c2128',
                               '#1c2128', '#1c2128', '#1c2128'])

        tbl = ax_table.table(
            cellText=rows,
            colLabels=['Time', 'Type', 'Entry', 'TP', 'SL', 'R:R', 'Score', 'HTF'],
            cellLoc='center',
            loc='upper center',
            bbox=[0, 0.02, 1, 0.97]
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(6.5)

        for (row_i, col_i), cell in tbl.get_celld().items():
            cell.set_edgecolor(COLORS['grid'])
            if row_i == 0:
                cell.set_facecolor('#30363d')
                cell.set_text_props(color=COLORS['text'], fontweight='bold')
            else:
                fc = col_colors[row_i - 1][col_i] if row_i - 1 < len(col_colors) else '#1c2128'
                cell.set_facecolor(fc)
                txt_c = COLORS['green'] if rows[row_i-1][1]=='BUY' and col_i==1 \
                        else COLORS['red'] if rows[row_i-1][1]=='SELL' and col_i==1 \
                        else COLORS['text']
                cell.set_text_props(color=txt_c)
    else:
        ax_table.text(0.5, 0.5, 'Belum ada history.\nJalankan analisis dulu.',
                      ha='center', va='center', color=COLORS['dim'],
                      fontsize=9, transform=ax_table.transAxes)

    # ── TITLE ─────────────────────────────────────────────────────────────────
    last_close = df['close'].iloc[-1]
    last_rsi   = df['rsi'].iloc[-1]
    trend_15m  = 'UPTREND 📈' if last_close > df['ema_trend'].iloc[-1] else 'DOWNTREND 📉'

    fig.suptitle(
        f"⚡ QUANTUM TRADING SYSTEM v2  |  {CONFIG['symbol']}  |  "
        f"${last_close:,.2f}  |  RSI {last_rsi:.1f}  |  15m: {trend_15m}  |  "
        f"HTF: {htf_bias}  |  News: {sentiment_label}\n"
        f"Updated: {now_str} WIB",
        color=COLORS['text'], fontsize=9.5, fontweight='bold', y=0.97, x=0.42
    )

    plt.show()
    return fig


print('✅ Chart engine ready.')

In [ ]:
# ─── CONSOLE DASHBOARD ──────────────────────────────────────────────────────

def print_dashboard(df, confirmed_signals, htf_bias, htf_reasons,
                    sentiment_label, news_list, news_summary, now_str):
    last = df.iloc[-1]
    trend  = '🟢 UPTREND' if last['close'] > last['ema_trend'] else '🔴 DOWNTREND'
    mom    = '▲ Bullish'  if last['macd'] > last['macd_signal'] else '▼ Bearish'
    htf_ic = '🟢' if htf_bias == 'BULLISH' else '🔴' if htf_bias == 'BEARISH' else '⚪'

    print('\033[1m' + '═' * 72 + '\033[0m')
    print(f"  ⚡ QUANTUM TRADING SYSTEM v2  |  {CONFIG['symbol']}  |  {now_str} WIB")
    print('═' * 72)
    print(f"  💰 Harga   : ${last['close']:>12,.4f} USDT")
    print(f"  📈 RSI     : {last['rsi']:>6.2f}  "
          f"{'🔴 OVERBOUGHT' if last['rsi'] > 70 else '🟢 OVERSOLD' if last['rsi'] < 30 else '⚪ NETRAL'}")
    print(f"  📊 MACD    : {last['macd']:>+10.4f}  →  {mom}")
    print(f"  〽️  ATR14  : {last['atr']:>10.4f}")
    print(f"  📦 Volume  : {last['volume']:>12,.2f}  × {last['vol_ratio']:.2f}")
    print(f"  🌊 BB Width: {last['bb_width']:.4f}  {'⚠️  SQUEEZE!' if last['bb_width'] < 0.02 else ''}")
    print(f"  🎯 15m     : {trend}")
    print('─' * 72)

    print(f"  {htf_ic} HTF BIAS: {htf_bias}")
    for r in htf_reasons:
        print(f"     • {r}")
    print('─' * 72)

    news_ic = '🚀' if 'VERY BULLISH' in sentiment_label else \
              '🟢' if 'BULLISH' in sentiment_label else \
              '💀' if 'VERY BEARISH' in sentiment_label else \
              '🔴' if 'BEARISH' in sentiment_label else '⚪'
    print(f"  {news_ic} NEWS SENTIMENT: {sentiment_label}")
    print(f"     {news_summary}")

    if news_list:
        print('  📰 Berita Terbaru (5):')    
        for n in news_list[:5]:
            ic = '🟢' if n['sentiment']=='BULLISH' else '🔴' if n['sentiment']=='BEARISH' else '⚪'
            hi = ' 💥' if n['high_impact'] else ''
            print(f"     {ic}{hi} [{n['age_hours']:.1f}j | {n['source']}] {n['title'][:70]}")
    print('─' * 72)

    buy_sigs  = [(ts, s) for ts, s in confirmed_signals if s['type'] == 'BUY']
    sell_sigs = [(ts, s) for ts, s in confirmed_signals if s['type'] == 'SELL']

    print('\n  🟢 SINYAL BUY TERKONFIRMASI:')
    if not buy_sigs:
        print('     — Tidak ada dalam window saat ini —')
    for ts, s in buy_sigs[-3:]:
        bar = '█' * s['score'] + '░' * (10 - s['score'])
        print(f"  {ts.strftime('%d/%m %H:%M')}  |  Entry: {s['close']:,.2f}  |  "
              f"TP: \033[92m{s['tp']:,.2f}\033[0m  |  SL: \033[91m{s['sl']:,.2f}\033[0m  |  "
              f"R:R {s['rr']:.1f}  |  [{bar}] {s['score']}/10")
        for r in s['reasons']:
            print(f"     • {r}")

    print('\n  🔴 SINYAL SELL TERKONFIRMASI:')
    if not sell_sigs:
        print('     — Tidak ada dalam window saat ini —')
    for ts, s in sell_sigs[-3:]:
        bar = '█' * s['score'] + '░' * (10 - s['score'])
        print(f"  {ts.strftime('%d/%m %H:%M')}  |  Entry: {s['close']:,.2f}  |  "
              f"TP: \033[92m{s['tp']:,.2f}\033[0m  |  SL: \033[91m{s['sl']:,.2f}\033[0m  |  "
              f"R:R {s['rr']:.1f}  |  [{bar}] {s['score']}/10")
        for r in s['reasons']:
            print(f"     • {r}")

    print('\n' + '═' * 72)


print('✅ Dashboard ready.')

In [ ]:
# ─── FETCH OHLCV HELPER ──────────────────────────────────────────────────────

def fetch_ohlcv(timeframe, limit):
    """Fetch OHLCV dari Bitget dan konversi ke DataFrame WIB."""
    raw = exchange.fetch_ohlcv(CONFIG['symbol'], timeframe=timeframe, limit=limit)
    df  = pd.DataFrame(raw, columns=['timestamp','open','high','low','close','volume'])
    df['timestamp'] = (
        pd.to_datetime(df['timestamp'], unit='ms')
        .dt.tz_localize('UTC')
        .dt.tz_convert(WIB)
    )
    df.set_index('timestamp', inplace=True)
    return df


print('✅ OHLCV fetcher ready.')

In [ ]:
# ─── MAIN ANALYSIS LOOP ─────────────────────────────────────────────────────

def fetch_and_analyze():
    """Core loop: fetch semua TF → news → indikator → sinyal → log → chart."""
    try:
        now_str = datetime.now(WIB).strftime('%Y-%m-%d %H:%M:%S')
        print(f"⏳ [{now_str} WIB] Memulai analisis...")

        # ── 1. Fetch OHLCV semua timeframe ────────────────────────────────────
        print('   📡 Mengambil data OHLCV (1D / 4H / 1H / 15m)...')
        df_daily = fetch_ohlcv(CONFIG['tf_daily'], CONFIG['limit_daily'])
        df_4h    = fetch_ohlcv(CONFIG['tf_4h'],    CONFIG['limit_4h'])
        df_1h    = fetch_ohlcv(CONFIG['tf_1h'],    CONFIG['limit_1h'])
        df_main  = fetch_ohlcv(CONFIG['tf_main'],  CONFIG['limit_main'])

        # ── 2. Fetch Berita ───────────────────────────────────────────────────
        print('   📰 Mengambil & menganalisis berita...')
        news_list = fetch_news()
        sentiment_label, news_score_bull, news_score_bear, news_summary = \
            aggregate_news_sentiment(news_list)

        # ── 3. Hitung Indikator (frame utama = 15m) ───────────────────────────
        df_main = compute_indicators(df_main)
        fib_levels, swing_high, swing_low = compute_fibonacci(df_main)

        # ── 4. Analisis Multi-Timeframe ───────────────────────────────────────
        htf_bias, htf_bull, htf_bear, htf_reasons = get_htf_trend(df_daily, df_4h, df_1h)

        # ── 5. Deteksi Sinyal (dengan konteks HTF + News) ─────────────────────
        confirmed_signals = detect_signals(
            df_main, fib_levels,
            htf_bias, htf_bull, htf_bear,
            news_score_bull, news_score_bear
        )

        # ── 6. Log sinyal baru ────────────────────────────────────────────────
        new_count = sum(1 for _, sig in confirmed_signals if HISTORY.add(sig))

        # ── 7. Tampilkan dashboard ────────────────────────────────────────────
        clear_output(wait=True)
        print_dashboard(
            df_main, confirmed_signals,
            htf_bias, htf_reasons,
            sentiment_label, news_list, news_summary, now_str
        )
        if new_count:
            print(f"  ✨ {new_count} sinyal baru disimpan ke history.")

        # ── 8. Plot chart ─────────────────────────────────────────────────────
        history_df = HISTORY.to_dataframe()
        plot_chart(
            df_main, confirmed_signals, fib_levels, history_df,
            news_list, sentiment_label, htf_bias, htf_reasons, now_str
        )

    except Exception as e:
        print(f"\n⚠️  Error: {e}")
        import traceback
        traceback.print_exc()
        time.sleep(15)


# ─── START ───────────────────────────────────────────────────────────────────
print('🚀 QUANTUM TRADING SYSTEM v2 STARTING...')
print(f"   Symbol     : {CONFIG['symbol']}")
print(f"   Timeframes : 1D → 4H → 1H → 15m")
print(f"   Min Score  : {CONFIG['min_score']}/10 (6 Teknikal + 2 MTF + 2 News)")
print(f"   News Window: {CONFIG['news_max_age_hours']} jam terakhir")
print(f"   Update     : setiap {CONFIG['update_sec']//60} menit")
print()

while True:
    fetch_and_analyze()
    next_run = datetime.now(WIB) + timedelta(seconds=CONFIG['update_sec'])
    print(f"\n  ⏳ Update berikutnya: {next_run.strftime('%H:%M:%S')} WIB")
    time.sleep(CONFIG['update_sec'])